# Лабораторная работа №1
## Разведочный анализ данных. Исследование и визуализация данных

**Цель работы:** изучение различных методов визуализации данных.

**Датасет:** FIFA 19 EDA Dataset (winterbreeze/fifa19eda)

## 1. Импорт необходимых библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Настройка стиля графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Библиотеки успешно импортированы")

## 2. Загрузка датасета

In [ ]:
# Загрузка датасета через kagglehub
path = kagglehub.dataset_download("winterbreeze/fifa19eda")
print("Path to dataset files:", path)

# Поиск CSV файла в загруженной директории
dataset_path = Path(path)
csv_files = list(dataset_path.glob("*.csv"))

if csv_files:
    csv_file = csv_files[0]
    print(f"Найден файл: {csv_file.name}")
    df = pd.read_csv(csv_file)
    print("Датасет успешно загружен!")
else:
    print("CSV файл не найден. Проверяем содержимое директории:")
    print(list(dataset_path.iterdir()))

## 3. Текстовое описание выбранного набора данных

### Описание датасета FIFA 19 EDA

Датасет FIFA 19 EDA содержит информацию о футболистах из популярной видеоигры FIFA 19. Этот датасет включает в себя различные характеристики игроков, такие как их навыки, физические параметры, позиции на поле, стоимость и заработная плата.

**Основные области применения:**
- Анализ характеристик футболистов
- Исследование взаимосвязей между различными навыками
- Построение моделей для предсказания стоимости игроков
- Кластеризация игроков по их характеристикам

**Особенности датасета:**
- Содержит информацию о большом количестве футболистов
- Включает как числовые, так и категориальные признаки
- Имеет различные метрики производительности игроков

## 4. Основные характеристики датасета

In [ ]:
# Первичный просмотр данных
print("=" * 60)
print("ОСНОВНЫЕ ХАРАКТЕРИСТИКИ ДАТАСЕТА")
print("=" * 60)

print(f"\n1. Размерность датасета:")
print(f"   Количество строк: {df.shape[0]}")
print(f"   Количество столбцов: {df.shape[1]}")

print(f"\n2. Названия столбцов:")
print(f"   Всего признаков: {len(df.columns)}")
print("\n   Первые 10 признаков:")
for i, col in enumerate(df.columns[:10], 1):
    print(f"   {i}. {col}")
if len(df.columns) > 10:
    print(f"   ... и еще {len(df.columns) - 10} признаков")

print(f"\n3. Типы данных:")
print(df.dtypes.value_counts())

print(f"\n4. Информация о пропущенных значениях:")
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Количество пропусков': missing_data,
    'Процент пропусков': missing_percent
})
missing_df = missing_df[missing_df['Количество пропусков'] > 0].sort_values('Количество пропусков', ascending=False)
if len(missing_df) > 0:
    print(missing_df.head(10))
else:
    print("   Пропущенных значений не обнаружено!")

In [ ]:
# Просмотр первых строк датасета
print("Первые 5 строк датасета:")
print("=" * 60)
df.head()

In [ ]:
# Статистическое описание числовых признаков
print("Статистическое описание числовых признаков:")
print("=" * 60)
df.describe()

In [ ]:
# Анализ категориальных признаков
print("Анализ категориальных признаков:")
print("=" * 60)

categorical_cols = df.select_dtypes(include=['object']).columns
print(f"\nКоличество категориальных признаков: {len(categorical_cols)}")

if len(categorical_cols) > 0:
    print("\nУникальные значения в категориальных признаках (первые 5):")
    for col in categorical_cols[:5]:
        unique_count = df[col].nunique()
        print(f"\n{col}:")
        print(f"  Количество уникальных значений: {unique_count}")
        if unique_count <= 20:
            print(f"  Значения: {df[col].unique()[:10].tolist()}")
        else:
            print(f"  Первые 10 значений: {df[col].unique()[:10].tolist()}")

# Визуализация пропущенных значений
if missing_data.sum() > 0:
    print("\n" + "=" * 60)
    print("ВИЗУАЛИЗАЦИЯ ПРОПУЩЕННЫХ ЗНАЧЕНИЙ")
    print("=" * 60)
    
    # Топ-10 признаков с пропусками
    top_missing = missing_df.head(10)
    
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(top_missing)), top_missing['Количество пропусков'], color='coral', alpha=0.7)
    plt.yticks(range(len(top_missing)), top_missing.index, fontsize=9)
    plt.xlabel('Количество пропусков', fontsize=11)
    plt.title('Топ-10 признаков с пропущенными значениями', fontsize=12, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    print("Рисунок 0: Визуализация пропущенных значений")

## 5. Визуальное исследование датасета

### 5.1. Распределение числовых признаков

In [ ]:
# Выбор числовых признаков для визуализации
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Исключаем ID и индексы, если они есть
numeric_cols = [col for col in numeric_cols if not any(x in col.lower() for x in ['id', 'index', 'unnamed'])]

print(f"Найдено {len(numeric_cols)} числовых признаков")
print(f"Первые 10: {numeric_cols[:10]}")

In [ ]:
# Гистограммы распределения для ключевых числовых признаков
key_features = numeric_cols[:12] if len(numeric_cols) >= 12 else numeric_cols

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.ravel()

for i, col in enumerate(key_features):
    if i < len(axes):
        df[col].hist(bins=50, ax=axes[i], edgecolor='black', alpha=0.7)
        axes[i].set_title(f'Распределение: {col}', fontsize=10, fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Частота')
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Гистограммы распределения числовых признаков', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("Рисунок 1: Гистограммы распределения числовых признаков")

### 5.2. Box-plot для выявления выбросов

In [ ]:
# Box-plot для ключевых признаков
key_features_box = numeric_cols[:8] if len(numeric_cols) >= 8 else numeric_cols

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, col in enumerate(key_features_box):
    if i < len(axes):
        box_plot = axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True)
        box_plot['boxes'][0].set_facecolor('lightblue')
        axes[i].set_title(f'Box-plot: {col}', fontsize=10, fontweight='bold')
        axes[i].set_ylabel('Значение')
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Box-plot для выявления выбросов', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("Рисунок 2: Box-plot для выявления выбросов в данных")

### 5.3. Анализ категориальных признаков

In [ ]:
# Визуализация категориальных признаков (топ-10 значений)
categorical_cols_vis = [col for col in categorical_cols if df[col].nunique() <= 20 and df[col].nunique() > 1][:4]

if len(categorical_cols_vis) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    for i, col in enumerate(categorical_cols_vis):
        if i < len(axes):
            value_counts = df[col].value_counts().head(10)
            axes[i].barh(range(len(value_counts)), value_counts.values, color='steelblue')
            axes[i].set_yticks(range(len(value_counts)))
            axes[i].set_yticklabels(value_counts.index, fontsize=8)
            axes[i].set_xlabel('Количество')
            axes[i].set_title(f'Топ-10 значений: {col}', fontsize=10, fontweight='bold')
            axes[i].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.suptitle('Распределение категориальных признаков', fontsize=14, fontweight='bold', y=1.02)
    plt.show()
    print("Рисунок 3: Распределение категориальных признаков")
else:
    print("Не найдено подходящих категориальных признаков для визуализации")

### 5.4. Парные зависимости между признаками

In [ ]:
# Scatter plot для пар ключевых признаков
if len(numeric_cols) >= 4:
    # Выбираем первые 4 числовых признака для парного анализа
    selected_features = numeric_cols[:4]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    pairs = [(0, 1), (0, 2), (1, 2), (2, 3)]
    
    for idx, (i, j) in enumerate(pairs):
        if idx < len(axes) and i < len(selected_features) and j < len(selected_features):
            axes[idx].scatter(df[selected_features[i]], df[selected_features[j]], 
                            alpha=0.5, s=20, color='coral')
            axes[idx].set_xlabel(selected_features[i], fontsize=9)
            axes[idx].set_ylabel(selected_features[j], fontsize=9)
            axes[idx].set_title(f'{selected_features[i]} vs {selected_features[j]}', 
                              fontsize=10, fontweight='bold')
            axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.suptitle('Парные зависимости между признаками', fontsize=14, fontweight='bold', y=1.02)
    plt.show()
    print("Рисунок 4: Scatter plot для анализа парных зависимостей")
    
# Дополнительная визуализация: Violin plot для сравнения распределений
if len(numeric_cols) >= 4:
    selected_for_violin = numeric_cols[:4]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    for idx, col in enumerate(selected_for_violin):
        if idx < len(axes):
            # Нормализуем данные для лучшей визуализации
            data = df[col].dropna()
            if len(data) > 0:
                axes[idx].violinplot([data], positions=[0], showmeans=True, showmedians=True)
                axes[idx].set_ylabel('Значение', fontsize=9)
                axes[idx].set_title(f'Violin plot: {col}', fontsize=10, fontweight='bold')
                axes[idx].set_xticks([0])
                axes[idx].set_xticklabels([col], fontsize=8, rotation=45, ha='right')
                axes[idx].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.suptitle('Violin plot для анализа распределений признаков', fontsize=14, fontweight='bold', y=1.02)
    plt.show()
    print("Рисунок 4a: Violin plot для анализа распределений")

## 6. Информация о корреляции признаков

### 6.1. Матрица корреляций

In [ ]:
# Вычисление корреляционной матрицы для числовых признаков
# Ограничиваемся первыми 15 признаками для читаемости
corr_features = numeric_cols[:15] if len(numeric_cols) >= 15 else numeric_cols

correlation_matrix = df[corr_features].corr()

# Визуализация корреляционной матрицы
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='coolwarm', center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8}, vmin=-1, vmax=1)
plt.title('Матрица корреляций между числовыми признаками', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("Рисунок 5: Тепловая карта корреляционной матрицы")

In [ ]:
# Поиск наиболее коррелированных пар признаков
print("=" * 60)
print("АНАЛИЗ КОРРЕЛЯЦИЙ")
print("=" * 60)

# Создаем список пар с высокой корреляцией
high_corr_pairs = []
for i in range(len(corr_features)):
    for j in range(i+1, len(corr_features)):
        corr_value = correlation_matrix.iloc[i, j]
        if abs(corr_value) > 0.7:  # Высокая корреляция
            high_corr_pairs.append({
                'Признак 1': corr_features[i],
                'Признак 2': corr_features[j],
                'Корреляция': corr_value
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Корреляция', key=abs, ascending=False)
    print(f"\nНайдено {len(high_corr_pairs)} пар с высокой корреляцией (|r| > 0.7):")
    print(high_corr_df.to_string(index=False))
else:
    print("\nПар с высокой корреляцией (|r| > 0.7) не найдено")

In [ ]:
# Визуализация топ-10 наиболее коррелированных пар
if len(high_corr_pairs) > 0:
    top_corr = high_corr_df.head(10)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    y_pos = np.arange(len(top_corr))
    colors = ['green' if x > 0 else 'red' for x in top_corr['Корреляция']]
    
    bars = ax.barh(y_pos, top_corr['Корреляция'], color=colors, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{row['Признак 1']} - {row['Признак 2']}" 
                        for _, row in top_corr.iterrows()], fontsize=9)
    ax.set_xlabel('Коэффициент корреляции', fontsize=11)
    ax.set_title('Топ-10 наиболее коррелированных пар признаков', 
                fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='x')
    
    # Добавляем значения на столбцы
    for i, (idx, row) in enumerate(top_corr.iterrows()):
        ax.text(row['Корреляция'], i, f"{row['Корреляция']:.3f}", 
               va='center', ha='left' if row['Корреляция'] > 0 else 'right',
               fontsize=8, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    print("Рисунок 6: Топ-10 наиболее коррелированных пар признаков")

### 6.2. Дополнительный анализ корреляций

In [ ]:
# Статистика по корреляциям
all_correlations = []
for i in range(len(corr_features)):
    for j in range(i+1, len(corr_features)):
        all_correlations.append(abs(correlation_matrix.iloc[i, j]))

corr_stats = pd.Series(all_correlations)

print("Статистика по абсолютным значениям корреляций:")
print("=" * 60)
print(f"Среднее значение: {corr_stats.mean():.3f}")
print(f"Медиана: {corr_stats.median():.3f}")
print(f"Минимальное значение: {corr_stats.min():.3f}")
print(f"Максимальное значение: {corr_stats.max():.3f}")
print(f"Стандартное отклонение: {corr_stats.std():.3f}")

# Гистограмма распределения корреляций
plt.figure(figsize=(10, 6))
plt.hist(all_correlations, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
plt.xlabel('Абсолютное значение корреляции', fontsize=11)
plt.ylabel('Частота', fontsize=11)
plt.title('Распределение абсолютных значений корреляций', fontsize=12, fontweight='bold')
plt.axvline(corr_stats.mean(), color='red', linestyle='--', linewidth=2, label=f'Среднее = {corr_stats.mean():.3f}')
plt.axvline(corr_stats.median(), color='green', linestyle='--', linewidth=2, label=f'Медиана = {corr_stats.median():.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nРисунок 7: Распределение абсолютных значений корреляций")

## 7. Выводы и заключение

### Основные результаты исследования:

1. **Характеристики датасета:**
   - Датасет содержит информацию о футболистах из FIFA 19
   - Размерность: [будет заполнено после выполнения]
   - Количество признаков: [будет заполнено после выполнения]

2. **Качество данных:**
   - [Информация о пропущенных значениях]
   - [Информация о выбросах]

3. **Визуальный анализ:**
   - Распределения признаков показывают [основные закономерности]
   - Выявлены выбросы в следующих признаках: [список]
   - Категориальные признаки демонстрируют [особенности распределения]

4. **Корреляционный анализ:**
   - Найдено [количество] пар признаков с высокой корреляцией
   - Средняя корреляция между признаками составляет [значение]
   - Наиболее сильные связи наблюдаются между: [примеры]

### Рекомендации для дальнейшего анализа:

1. Обработка пропущенных значений (если они есть)
2. Нормализация признаков для моделей машинного обучения
3. Удаление или трансформация сильно коррелированных признаков
4. Дополнительный анализ выбросов и их влияние на модели

## 8. Список использованных инструментов и библиотек

- **Python 3.x** - язык программирования
- **Pandas** - для работы с данными и DataFrame
- **NumPy** - для численных вычислений
- **Matplotlib** - для создания графиков и визуализаций
- **Seaborn** - для расширенных визуализаций и статистических графиков
- **Kagglehub** - для загрузки датасетов с Kaggle
- **Jupyter Notebook** - среда для интерактивной разработки

---

**Примечание для отчета:** 

После выполнения всех ячеек ноутбука, скопируйте полученные результаты и графики в документ Word. Каждый рисунок должен быть подписан согласно указанным номерам (Рисунок 0, Рисунок 1, Рисунок 2 и т.д.). 

Для сохранения графиков в высоком качестве используйте:
```python
plt.savefig('figure_name.png', dpi=300, bbox_inches='tight')
```

**Структура отчета должна включать:**
1. Титульный лист
2. Описание задания
3. Текст программы (ячейки кода из ноутбука)
4. Экранные формы с примерами выполнения (результаты выполнения ячеек и графики)